# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:

%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371


In [5]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435,8.0,0.136531,0.239391,1223.0,1353.0,0.903917
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875,38.0,0.146092,0.336034,900.0,2173.0,0.414174
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556,42.0,0.064587,0.803004,103.0,1146.0,0.089878
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698,4.0,0.384248,0.408115,38.0,87.0,0.436782
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371,3.0,0.104607,0.860845,15.0,36.0,0.416667


In [7]:
print(data.columns)
print(data.shape)
print(data.columns.tolist())

Index(['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30',
       'clk_last30', 'pos_last30', 'visible_queries', 'rare_share',
       'anon_share', 'top_query_impressions', 'kept_impressions',
       'top_query_share'],
      dtype='object')
(111247, 12)
['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I selected Random Forest because it can capture non-linear relationships between search performance features and the proxy decline label. It also provides feature importance, making the model easier to interpret. This method is suitable for the Growth / Recovery / Momentum Prediction lane and can be directly compared with the Week-4 baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used GroupShuffleSplit grouped by client_hash_id so that content from the same client does not appear in both training and test sets. This produces a more honest evaluation because the model is tested on unseen clients rather than memorizing client-specific patterns.

In [17]:
from sklearn.model_selection import GroupShuffleSplit

groups = data["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

##Creating Proxy Label:
I created a proxy label based on the change in impressions over two consecutive 30-day windows.

Proxy label: is_declining = (imp_last30 < 0.8 × imp_prev30)

Since imp_last30 is part of the proxy label definition, it was excluded from the final feature set to avoid data leakage.

In [8]:
# Create proxy label
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

# Feature matrix
X = data[
    [
        "imp_prev30",
        "clk_last30",
        "pos_last30",
        "visible_queries",
        "rare_share",
        "anon_share",
        "top_query_share",
    ]
]

# Target
y = data["is_declining"]

In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [11]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [12]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", accuracy)

Random Forest Accuracy: 0.8157492910637679


In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.67      0.72      4765
           1       0.84      0.89      0.86      8988

    accuracy                           0.82     13753
   macro avg       0.80      0.78      0.79     13753
weighted avg       0.81      0.82      0.81     13753



In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[3213 1552]
 [ 982 8006]]


In [15]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.8649921593020612


## Comparison with Week-4 Baseline

The Week-4 baseline ranked pages using fixed rules based on impression drop and top query concentration.

In this assignment, I trained a Random Forest classifier using the same warehouse dataset and the same proxy label (`is_declining`).

Unlike the baseline, the Random Forest learns relationships from multiple search signals instead of relying on manually defined rules.

### Results

| Method | Result |
|---------|--------|
| Week-4 Baseline | Rule-based ranking |
| Week-5 Model | Random Forest |
| Accuracy | 81.6% |
| ROC-AUC | 0.865 |

The Random Forest achieved good predictive performance while using only observable search signals. This suggests that a machine learning model can capture more complex decline patterns than a simple rule-based baseline.

Both the baseline and the model were built using the same dataset and the same proxy label, making the comparison fair.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest model achieved good overall performance, but it still makes both false positive and false negative errors.

False positives occur when the model predicts a page is declining although the proxy label says it is not.

False negatives occur when the model misses some declining pages.

Feature importance shows that the model relies mainly on average position, previous impressions, and recent clicks. Query-level features contribute additional information but are less influential.

Overall, the model captures the main decline patterns while remaining suitable as a decision-support tool rather than a final decision maker.

In [19]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print(cm)

error_summary = pd.DataFrame({
    "Metric": [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive"
    ],
    "Count": [
        cm[0,0],
        cm[0,1],
        cm[1,0],
        cm[1,1]
    ]
})

print(error_summary)

feature_importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print("\nFeature Importance")
print(feature_importance)


Confusion Matrix
[[3213 1552]
 [ 982 8006]]
           Metric  Count
0   True Negative   3213
1  False Positive   1552
2  False Negative    982
3   True Positive   8006

Feature Importance
           Feature  Importance
2       pos_last30    0.206305
0       imp_prev30    0.191672
1       clk_last30    0.140535
4       rare_share    0.136840
5       anon_share    0.134058
6  top_query_share    0.108884
3  visible_queries    0.081705


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.